# Composer v2 全量官谱训练
使用 GPU 运行时。先将 composer_v2_full.tar.gz 和对应 sha256 上传到 Drive 的 Composer_V2_Full 目录。运行本笔记本将准备 MERT 特征，首次可能较慢。模型使用单卡 BF16，训练总目标 50 轮。
MERT 缓存在 Colab 本地磁盘。运行时重建后需要重新准备，检查点保存在 Drive。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/Composer_V2_Full')
ARCHIVE=BASE/'composer_v2_full.tar.gz'
BUNDLE=Path('/content/composer_v2_full')
RUN=BASE/'run'
CACHE=Path('/content/composer_v2_cache')


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), '请选择支持 BF16 的 GPU'
print(torch.cuda.get_device_name())


In [ ]:
import hashlib, shutil, tarfile
assert shutil.disk_usage('/content').free >= 60 * 2**30, '请使用至少 60 GiB 可用本地磁盘的运行时'
assert ARCHIVE.is_file(), f'请上传 {ARCHIVE}'
expected=Path(str(ARCHIVE)+'.sha256').read_text().split()[0]
local=Path('/content/composer_v2_full.tar.gz')
shutil.copyfile(ARCHIVE,local)
h=hashlib.sha256()
with local.open('rb') as f:
    for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
assert h.hexdigest()==expected, '文件校验失败'
assert not BUNDLE.exists(), '已解压过，跳过此格'
BUNDLE.mkdir()
with tarfile.open(local) as t: t.extractall(BUNDLE,filter='data')
local.unlink()


In [ ]:
import os, json, sys, subprocess
PROJECT=BUNDLE/'project'
subprocess.run([sys.executable,'-m','pip','install','-e',str(PROJECT)+'[composer-v2-music]'],check=True)
mapfile=BUNDLE/'path_map.json'
mapfile.write_text(json.dumps({'E:/':str(BUNDLE/'materials/E')},ensure_ascii=False),encoding='utf-8')
os.environ['MAICHART_PATH_MAP']=str(mapfile)
RUN.mkdir(parents=True,exist_ok=True)
CACHE.mkdir(parents=True,exist_ok=True)


In [ ]:
# 首次运行新建训练；重新连接后从 Drive 的完整检查点续训。
args=[sys.executable,'-u','-m','maichart.composer_v2.train',
 '--manifest',str(BUNDLE/'splits/train_manifest.json'),
 '--validation-manifest',str(BUNDLE/'splits/dev_manifest.json'),
 '--checkpoint-dir',str(RUN),'--cache-dir',str(CACHE),
 '--epochs','50','--batch-size','8','--grad-accumulation','8',
 '--lr','0.0002','--lr-schedule','constant','--warmup-steps','200',
 '--precision','bf16','--device','cuda','--prepare-device','cuda',
 '--sequence-policy','continuous_events_v1','--refresh-events','16',
 '--condition-policy','radar_style_v1','--cpu-threads','4',
 '--checkpoint-interval-steps','100','--validation-max-batches','0']
last=RUN/'composer_v2_last.pt'
if last.exists(): args += ['--resume',str(last)]
subprocess.run(args,cwd=PROJECT,check=True)
